In [3]:
import pandas as pd
import numpy as np
import os

# ================================
# GENERATE SPLIT FILES FOR YOUR PROJECT
# ================================

print("🔄 Loading your battery data...")

# Load your processed battery data
try:
    df = pd.read_csv('../DATA/Battery_RUL_with_features.csv')
    print(f"✅ Loaded data successfully!")
    print(f"📊 Data shape: {df.shape}")
    print(f"📋 Columns: {list(df.columns)}")
except FileNotFoundError:
    print("❌ Could not find '../DATA/Battery_RUL_with_features.csv'")
    print("Please make sure the file exists and the path is correct.")
    exit()

# ================================
# PERFORM CYCLE-BASED SPLIT
# ================================

print("\n🔀 Performing cycle-based split (70/15/15)...")

# Sort by cycle index to maintain chronological order
df_sorted = df.sort_values('Cycle_Index').reset_index(drop=True)
n_samples = len(df_sorted)

print(f"📊 Total samples: {n_samples}")
print(f"🔋 Cycle range: {df_sorted['Cycle_Index'].min()} to {df_sorted['Cycle_Index'].max()}")

# Calculate split points
train_end = int(n_samples * 0.7)
val_end = int(n_samples * 0.85)  # 0.7 + 0.15

# Create splits
train_df = df_sorted.iloc[:train_end].copy()
val_df = df_sorted.iloc[train_end:val_end].copy()
test_df = df_sorted.iloc[val_end:].copy()

print(f"📈 Train: {len(train_df)} samples (cycles {train_df['Cycle_Index'].min()}-{train_df['Cycle_Index'].max()})")
print(f"📊 Val:   {len(val_df)} samples (cycles {val_df['Cycle_Index'].min()}-{val_df['Cycle_Index'].max()})")
print(f"📉 Test:  {len(test_df)} samples (cycles {test_df['Cycle_Index'].min()}-{test_df['Cycle_Index'].max()})")

# ================================
# PREPARE FEATURES AND TARGETS
# ================================

print("\n🎯 Preparing features and targets...")

# Define feature columns (exclude target and metadata columns)
exclude_cols = [
    'SOH',                    # Target variable
    'RUL',                    # Alternative target
    'Cycle_Index',            # Just an index
    'cell',                   # Cell identifier (if exists)
    'cycle',                  # Alternative cycle name (if exists)
    'Discharge_Time_Norm'     # Might be redundant
]

# Get feature columns that actually exist in your data
available_features = [col for col in df.columns if col not in exclude_cols]
print(f"🔧 Available features: {available_features}")

# Choose target (you can change this to 'RUL' if preferred)
target_column = 'SOH'
print(f"🎯 Target variable: {target_column}")

# Create feature matrices (X) and target vectors (y)
X_train = train_df[available_features].copy()
y_train = train_df[target_column].copy()

X_val = val_df[available_features].copy()
y_val = val_df[target_column].copy()

X_test = test_df[available_features].copy()
y_test = test_df[target_column].copy()

print(f"✅ Features prepared: {X_train.shape[1]} features")
print(f"✅ Train set: X{X_train.shape}, y{y_train.shape}")
print(f"✅ Val set:   X{X_val.shape}, y{y_val.shape}")
print(f"✅ Test set:  X{X_test.shape}, y{y_test.shape}")

# ================================
# CREATE SPLITS DIRECTORY AND SAVE FILES
# ================================

print("\n💾 Saving split files...")

# Create splits directory
splits_dir = '../DATA/splits'
os.makedirs(splits_dir, exist_ok=True)
print(f"📁 Created directory: {splits_dir}")

# Save feature files (X_train, X_val, X_test)
X_train.to_csv(f'{splits_dir}/X_train.csv', index=False)
X_val.to_csv(f'{splits_dir}/X_val.csv', index=False)
X_test.to_csv(f'{splits_dir}/X_test.csv', index=False)

print("✅ Saved feature files:")
print(f"   - X_train.csv ({X_train.shape[0]} rows, {X_train.shape[1]} features)")
print(f"   - X_val.csv ({X_val.shape[0]} rows, {X_val.shape[1]} features)")
print(f"   - X_test.csv ({X_test.shape[0]} rows, {X_test.shape[1]} features)")

# Save target files (y_train, y_val, y_test)
# Save as DataFrames to maintain column names
pd.DataFrame(y_train).to_csv(f'{splits_dir}/y_train.csv', index=False)
pd.DataFrame(y_val).to_csv(f'{splits_dir}/y_val.csv', index=False)
pd.DataFrame(y_test).to_csv(f'{splits_dir}/y_test.csv', index=False)

print("✅ Saved target files:")
print(f"   - y_train.csv ({len(y_train)} values)")
print(f"   - y_val.csv ({len(y_val)} values)")  
print(f"   - y_test.csv ({len(y_test)} values)")

# ================================
# VERIFICATION AND SUMMARY
# ================================

print("\n🔍 Verification...")

# Check for data leakage
train_cycles = set(train_df['Cycle_Index'])
val_cycles = set(val_df['Cycle_Index'])
test_cycles = set(test_df['Cycle_Index'])

overlaps = len(train_cycles.intersection(val_cycles)) + len(train_cycles.intersection(test_cycles)) + len(val_cycles.intersection(test_cycles))

if overlaps == 0:
    print("✅ No data leakage detected - splits are clean!")
else:
    print(f"⚠️ Warning: {overlaps} overlapping cycles detected")

# Show file sizes
print(f"\n📊 Generated files in {splits_dir}/:")
for filename in ['X_train.csv', 'X_val.csv', 'X_test.csv', 'y_train.csv', 'y_val.csv', 'y_test.csv']:
    filepath = f'{splits_dir}/{filename}'
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"   📄 {filename}: {size_kb:.1f} KB")

# Show feature names for reference
print(f"\n🔧 Feature columns used:")
for i, feature in enumerate(available_features, 1):
    print(f"   {i:2d}. {feature}")

print(f"\n🎯 Target column: {target_column}")

print("\n" + "="*50)
print("🎉 SUCCESS! All split files generated!")
print("="*50)
print("✅ You now have:")
print("   - X_train.csv, X_val.csv, X_test.csv (feature matrices)")
print("   - y_train.csv, y_val.csv, y_test.csv (target vectors)")
print("   - 70/15/15 chronological split")
print("   - No data leakage")
print("   - Ready for machine learning!")

# ================================
# BONUS: QUICK PREVIEW OF EACH FILE
# ================================

print("\n📋 QUICK PREVIEW OF GENERATED FILES:")
print("-" * 50)

files_to_preview = {
    'X_train.csv': 'Training Features',
    'X_val.csv': 'Validation Features', 
    'X_test.csv': 'Test Features',
    'y_train.csv': 'Training Targets',
    'y_val.csv': 'Validation Targets',
    'y_test.csv': 'Test Targets'
}

for filename, description in files_to_preview.items():
    filepath = f'{splits_dir}/{filename}'
    if os.path.exists(filepath):
        preview_df = pd.read_csv(filepath)
        print(f"\n📄 {description} ({filename}):")
        print(f"   Shape: {preview_df.shape}")
        print(f"   First few values:")
        if preview_df.shape[1] == 1:  # Target files
            print(f"   {list(preview_df.iloc[:5, 0].values)}")
        else:  # Feature files
            print(f"   Columns: {list(preview_df.columns)}")
            print(f"   Sample: {preview_df.iloc[0].to_dict()}")

print("\n🚀 You can now proceed to model training!")
print("💡 Use these files in your ML notebooks!")

🔄 Loading your battery data...
✅ Loaded data successfully!
📊 Data shape: (15064, 15)
📋 Columns: ['Cycle_Index', 'Discharge Time (s)', 'Decrement 3.6-3.4V (s)', 'Max. Voltage Dischar. (V)', 'Min. Voltage Charg. (V)', 'Time at 4.15V (s)', 'Time constant current (s)', 'Charging time (s)', 'RUL', 'Voltage_Range', 'Discharge_Time_Norm', 'SOH', 'Delta_SOH', 'Rolling_DischargeTime', 'Rolling_SOH']

🔀 Performing cycle-based split (70/15/15)...
📊 Total samples: 15064
🔋 Cycle range: 1 to 1134
📈 Train: 10544 samples (cycles 1-777)
📊 Val:   2260 samples (cycles 777-949)
📉 Test:  2260 samples (cycles 949-1134)

🎯 Preparing features and targets...
🔧 Available features: ['Discharge Time (s)', 'Decrement 3.6-3.4V (s)', 'Max. Voltage Dischar. (V)', 'Min. Voltage Charg. (V)', 'Time at 4.15V (s)', 'Time constant current (s)', 'Charging time (s)', 'Voltage_Range', 'Delta_SOH', 'Rolling_DischargeTime', 'Rolling_SOH']
🎯 Target variable: SOH
✅ Features prepared: 11 features
✅ Train set: X(10544, 11), y(10544